# Amaliy mashgʻulot 13. Avtomatik referatlash: ekstraktiv (TextRank) va abstraktiv usullar

13-laboratoriyada gaplar oʻxshashligi grafini qurdingiz, PageRank iteratsiyalari muhim tugunlarni qanday «kattalashtirishini» koʻrdingiz va ekstraktiv referatni abstraktiv referat bilan solishtirdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. Samarqand haqidagi matnni gaplarga ajratamiz va soʻz oʻzaklari «qoplarini» tayyorlaymiz;
2. gaplar oʻxshashligi grafini quramiz, PageRank ni darajali usul bilan noldan hisoblaymiz va `networkx.pagerank` bilan solishtiramiz;
3. ekstraktiv referat tuzamiz va uni qoʻlda hisoblangan ROUGE-1 metrikasi bilan baholaymiz;
4. kalit soʻzlarni topish uchun TextRank ni soʻzlarga qoʻllaymiz;
5. ekstraktiv referatni abstraktiv referat bilan solishtiramiz va pointer-generator gʻoyasini (koʻchirish yoki yaratish) koʻrib chiqamiz.

*«NLP algoritmlari» kursining 13-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Matn va gaplarga ajratish

Ekstraktiv referat asl matndagi butun gaplardan tuziladi, shuning uchun avval matnni toʻgʻri boʻlish kerak. Oddiy qoida: gap `.`, `!`, `?` bilan tugaydi, ulardan keyin boʻsh joy va bosh harf yoki raqam keladi. Quyida bizning matn (13 gap) va bu qoida buziladigan misol: «y.» (yil) qisqartmasi va ism-sharifning bosh harflari.

In [ ]:
text = """Samarqand — Markaziy Osiyodagi eng qadimiy shaharlardan biri, uning yoshi 2700 yildan oshadi. Qadimgi yunon manbalarida shahar Marakanda deb atalgan. Samarqand Buyuk ipak yoʻlida joylashgan, shahar orqali Xitoydan Eronga va undan ham uzoqqa karvonlar oʻtgan. XIV asrda Amir Temur Samarqandni oʻz davlatining poytaxtiga aylantirdi. Temur davrida Samarqandda Bibixonim masjidi va boshqa koʻplab binolar qurildi. Temurning nabirasi Mirzo Ulugʻbek ham hukmdor, ham astronom olim edi. 1420-yillarda Ulugʻbek Samarqandda rasadxona qurdirdi. Rasadxona astronomlari mingdan ortiq yulduz katalogini tuzdilar. Registon maydonini uchta madrasa oʻrab turadi: Ulugʻbek, Sherdor va Tillakori madrasalari. Ulugʻbek madrasasi Registondagi eng qadimiy bino boʻlib, u XV asrda qurilgan. Amir Temur Goʻri Amir maqbarasida dafn etilgan. 2001-yilda Samarqandning tarixiy markazi YUNESKO Butunjahon merosi roʻyxatiga kiritildi. Har yili minglab sayyohlar Registon, Goʻri Amir va Shohi Zinda majmuasini koʻrish uchun Samarqandga keladi."""

import re
import numpy as np

def split_sentences(text):
    # . ! ? … dan keyin boʻsh joy va bosh harf, raqam yoki qoʻshtirnoq boʻlsa, kesamiz
    return [s.strip() for s in re.split(r"(?<=[.!?…])\s+(?=[«\"A-ZА-ЯЁ0-9])", text.strip()) if s.strip()]

sents = split_sentences(text)
for i, s in enumerate(sents):
    print(f"{i:>2}. {s}")
print("\njami gaplar:", len(sents))

print("\ntuzoq: «y.» qisqartmasi va ism-sharifning bosh harflari:")
for s in split_sentences("1417-y. Ulugʻbek madrasa qurdira boshladi. V. V. Bartold «Ulugʻbek va uning davri» kitobini yozgan."):
    print("   |", s)

## 2. Tayyorlash: yordamchi soʻzlarsiz soʻz oʻzaklari

Gaplar oʻxshashligini umumiy soʻzlar boʻyicha hisoblaymiz. «Samarqand», «Samarqandda», «Samarqandni» bitta soʻz deb hisoblanishi uchun soʻzlarni oʻzakka keltiramiz. Oʻzbek tili uchun tayyor stemmerlar kam (`nltk` dagi Snowball da oʻzbek tili yoʻq), shuning uchun 11-amaliy mashgʻulotdagi kabi oddiy qoidaviy stemmer yozamiz: soʻz oxiridan kelishik, egalik va koʻplik qoʻshimchalarini olib tashlaymiz. Stemmer qoʻpol: u feʼllardagi «-i» ni ham kesadi (qurildi → qurild) va baʼzan atoqli otlarni buzadi (Marakanda → marakan, Bibixonim → bibixon), lekin bizga asosan bitta soʻzning shakllari bitta oʻzakka tushishi muhim. Yordamchi soʻzlarni (bogʻlovchilar, olmoshlar, koʻmakchilar) toʻxtash soʻzlari roʻyxati bilan olib tashlaymiz: ular deyarli har bir gapda bor va mavzuni bildirmaydi.

In [ ]:
def tokens(text):
    return re.findall(r"[а-яёa-z][а-яёa-zʻʼ'-]*", text.lower())

# qoʻshimchalar guruhlari soʻz oxiridan boshlab: kelishik -> egalik -> koʻplik
CASE = ["ning", "dagi", "dan", "da", "ga", "ka", "qa", "ni"]
POSS = ["imiz", "ingiz", "lari", "im", "ing", "si", "i"]
PLUR = ["lar"]

def stem(word, min_len=3):
    for group in (CASE, POSS, PLUR):
        for suf in group:                      # har bir guruhdan koʻpi bilan bitta qoʻshimcha
            if word.endswith(suf) and len(word) - len(suf) >= min_len:
                word = word[: -len(suf)]
                break
    return word

STOP = set("""va bilan uchun ham bu shu u ular uning ularning edi oʻz eng biri deb orqali har undan
    boʻlib bir boshqa koʻplab yoki lekin ammo esa""".split())

def content_words(sentence):
    # toʻxtash soʻzlari va bir harfli boʻlaklarsiz soʻz oʻzaklari
    return [stem(w) for w in tokens(sentence) if w not in STOP and len(w) > 1]

bags = [content_words(s) for s in sents]
for i, b in enumerate(bags[:5]):
    print(f"{i:>2}.", " ".join(b))
print("...")

## 3. Gaplar oʻxshashligi grafi

Graf tugunlari — gaplar, qirra vazni — oʻxshashlik. TextRank haqidagi asl maqolada (Mihalcea, Tarau, 2004) ikki gapning oʻxshashligi — umumiy soʻzlar soni ularning uzunliklari logarifmlari yigʻindisiga boʻlingan:

$$w_{ij} = \frac{|\{w : w \in S_i \text{ va } w \in S_j\}|}{\log |S_i| + \log |S_j|}$$

Maxrajdagi logarifm uzun gaplarning faqat uzunligi hisobiga yutishiga yoʻl qoʻymaydi.

In [ ]:
import math
import pandas as pd

def overlap_similarity(a, b):
    # TextRank maqolasidagi oʻxshashlik: umumiy soʻzlar / (log|S_i| + log|S_j|)
    common = len(set(a) & set(b))
    if common == 0 or len(a) < 2 or len(b) < 2:
        return 0.0
    return common / (math.log(len(a)) + math.log(len(b)))

n = len(sents)
W = np.array([[overlap_similarity(bags[i], bags[j]) if i != j else 0.0 for j in range(n)] for i in range(n)])
print(pd.DataFrame(W).round(2).to_string())
print(f"\nvazni > 0 boʻlgan qirralar: {int((W > 0).sum() / 2)} / {n * (n - 1) // 2}")
i, j = 8, 9
print(f"misol, gaplar {i}–{j}: {sorted(set(bags[i]) & set(bags[j]))}, "
      f"|S{i}| = {len(bags[i])}, |S{j}| = {len(bags[j])}, w = {W[i, j]:.3f}")

## 4. PageRank darajali usul bilan

Muhim gap boshqa muhim gaplarga oʻxshaydi. PageRank bu gʻoyani formulaga aylantiradi: «tasodifiy oʻquvchi» qirralar boʻylab gapdan gapga oʻtadi (vaznga proporsional ehtimol bilan), $1-d$ ehtimol bilan esa tasodifiy gapga sakraydi:

$$PR(i) = \frac{1-d}{N} + d \sum_{j} \frac{w_{ji}}{\sum_k w_{jk}}\, PR(j), \qquad d = 0.85$$

$PR = 1/N$ dan boshlaymiz va qiymatlar oʻzgarmay qolguncha formulani takrorlaymiz (darajali usul). Quyida dastlabki iteratsiyalar va oxirgisi chop etiladi.


`networkx.pagerank` bilan solishtiramiz. TextRank maqolasida formula $N$ ga boʻlmasdan yozilgan ($\frac{1-d}{N}$ oʻrniga $1-d$); bu faqat masshtab, gaplar tartibi bir xil.

In [ ]:
def pagerank(W, d=0.85, tol=1e-10, max_iter=100, verbose=False):
    n = len(W)
    out = W.sum(axis=1)
    M = np.divide(W, out[:, None], out=np.zeros_like(W), where=out[:, None] > 0).T   # M[i, j] = w_ji / sum_k w_jk: j tugun i ga beradigan vazn ulushi
    dangling = out == 0                                                              # qirrasiz tugunlar vaznini hammaga teng boʻlib beradi (networkx dagidek)
    p = np.full(n, 1 / n)
    history = [p]
    for it in range(1, max_iter + 1):
        p_new = (1 - d) / n + d * (M @ p + p[dangling].sum() / n)
        delta = np.abs(p_new - p).sum()
        p = p_new
        history.append(p)
        if verbose and (it <= 3 or delta < tol):
            print(f"iteratsiya {it:>2}: delta = {delta:.2e}   " + " ".join(f"{x:.3f}" for x in p))
        if delta < tol:
            break
    return p, history

scores, history = pagerank(W, verbose=True)

import networkx as nx

G = nx.from_numpy_array(W)                          # W matritsadan yoʻnalishsiz vaznli graf
pr = nx.pagerank(G, alpha=0.85, weight="weight", tol=1e-12)
nx_scores = np.array([pr[i] for i in range(n)])
print(f"max |bizning PageRank - networkx| = {np.abs(scores - nx_scores).max():.2e}")

order = np.argsort(-scores)
print("\nPageRank kamayishi boʻyicha gaplar:")
for r, i in enumerate(order, start=1):
    print(f"{r:>2}. [{i:>2}] {scores[i]:.3f}  {sents[i][:70]}")

Chapda — graf: tugun hajmi PageRank ga, qirra qalinligi vaznga proporsional, eng yaxshi uchta gap toʻq sariq rangda. Oʻngda — har bir gapning PageRank qiymati iteratsiyalar boʻyicha qanday oʻzgarishi. Usul bir necha oʻnta qadamda yaqinlashadi.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.4, 1]})
pos = nx.circular_layout(G)
widths = [3 * G[u][v]["weight"] for u, v in G.edges()]
top = set(order[:3])
nx.draw_networkx_edges(G, pos, ax=ax[0], width=widths, alpha=0.35)
nx.draw_networkx_nodes(G, pos, ax=ax[0], node_size=[12000 * s for s in scores],
                       node_color=["#D9480F" if i in top else "#2B55C9" for i in range(n)])
nx.draw_networkx_labels(G, pos, ax=ax[0], font_color="white", font_size=9)
ax[0].set_title("Gaplar grafi (hajm = PageRank)")
ax[0].axis("off")

H = np.array(history)
for i in range(n):
    ax[1].plot(H[:, i], color="#D9480F" if i in top else "#9AA5B1", lw=2 if i in top else 1)
ax[1].set_xlabel("iteratsiya")
ax[1].set_ylabel("PageRank")
ax[1].set_title("PageRank yaqinlashishi")
plt.tight_layout()
plt.show()

## 5. Ekstraktiv referat va ROUGE-1

Referat — PageRank i eng katta boʻlgan $k$ ta gap, **asl matndagi tartibda** (aks holda matnni oʻqib boʻlmaydi). Siqish darajasini laboratoriyadagi surgich kabi `ratio` gaplar ulushi bilan beramiz.


Ikkinchi savol — oʻxshashlik oʻlchovini tanlash muhimmi? TextRank formulasini TF-IDF kosinusi bilan (11-amaliy mashgʻulotdan) almashtiramiz va qaysi gaplar referatga tushishini solishtiramiz.

In [ ]:
def summarize(sents, scores, ratio=0.25):
    # eng yaxshi k ta gap, lekin asl matn tartibida
    k = max(1, round(ratio * len(sents)))
    chosen = sorted(int(i) for i in np.argsort(-scores)[:k])
    return [sents[i] for i in chosen], chosen

summary, chosen = summarize(sents, scores, ratio=0.25)
print(f"tanlangan gaplar: {chosen}\n")
for i, s in enumerate(sents):
    mark = ">>" if i in chosen else "  "
    print(f"{mark} {i:>2}. {s}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(analyzer=lambda s: s)          # tayyor oʻzaklar roʻyxatlarini vektorlaymiz
C = cosine_similarity(vec.fit_transform(bags))
np.fill_diagonal(C, 0)
cos_scores, _ = pagerank(C)
print("TextRank formulasi boʻyicha uchtalik:", sorted(int(i) for i in np.argsort(-scores)[:3]))
print("TF-IDF kosinusi boʻyicha uchtalik:    ", sorted(int(i) for i in np.argsort(-cos_scores)[:3]))
print(f"ikki PageRank baholari toʻplamining korrelyatsiyasi: {np.corrcoef(scores, cos_scores)[0, 1]:.2f}")

Referatni qanday baholash mumkin? Inson yozgan namuna kerak. **ROUGE-1** metrikasi nomzod va namunadagi mos keladigan soʻzlarni (unigrammlarni) sanaydi: toʻliqlik $R$ = mosliklar / namuna uzunligi, aniqlik $P$ = mosliklar / nomzod uzunligi, $F_1$ — ularning garmonik oʻrtachasi. ROUGE va BLEU ni 18-amaliy mashgʻulotda batafsil koʻramiz.

TextRank ni ikkita oddiy bazaviy usul bilan solishtiramiz: **Lead-k** (dastlabki $k$ ta gap, yangiliklar uchun kuchli bazaviy usul) va tasodifiy $k$ ta gap. ROUGE-1 ni soʻz shakllari va oʻzaklar boʻyicha hisoblaymiz.

In [ ]:
# inson yozgan namunaviy referat (abstraktiv: oʻz soʻzlari bilan)
reference = ("Samarqand — Buyuk ipak yoʻlidagi qadimiy shahar, uning tarixi 2700 yildan ortiq. "
             "Amir Temur uni poytaxtga aylantirgan, nabirasi Ulugʻbek esa bu yerda rasadxona qurdirgan. "
             "Registon maydoni joylashgan tarixiy markaz YUNESKO Butunjahon merosi roʻyxatiga kiritilgan.")

from collections import Counter

def rouge1(candidate, reference, prep):
    c, r = Counter(prep(candidate)), Counter(prep(reference))
    overlap = sum((c & r).values())                # har bir soʻz namunada necha marta boʻlsa, koʻpi bilan shuncha marta hisoblanadi
    p = overlap / max(1, sum(c.values()))
    rec = overlap / max(1, sum(r.values()))
    f = 2 * p * rec / (p + rec) if p + rec else 0.0
    return p, rec, f

k = len(chosen)
rng = np.random.default_rng(0)
candidates = {
    "TextRank": " ".join(summary),
    f"Lead-{k}": " ".join(sents[:k]),
    "tasodifiy": " ".join(sents[i] for i in sorted(rng.choice(n, size=k, replace=False))),
}
print(f"{'':<12}{'soʻz shakllari':>22}{'oʻzaklar':>26}")
print(f"{'':<12}{'P':>8}{'R':>7}{'F1':>7}{'P':>9}{'R':>7}{'F1':>7}")
for name, cand in candidates.items():
    a = rouge1(cand, reference, tokens)
    b = rouge1(cand, reference, lambda t: [stem(w) for w in tokens(t)])
    print(f"{name:<12}" + "".join(f"{x:7.2f}" for x in a) + "  " + "".join(f"{x:7.2f}" for x in b))

Bunday kichik matnda usullar orasidagi farq katta emas, bitta songa ortiqcha baho bermaslik kerak: namuna bitta, ROUGE esa maʼnoni emas, faqat soʻzlar mosligini koʻradi. Eʼtibor bering: oʻzaklar boʻyicha solishtirish baholarni oshiradi — «shaharlardan» va «shahar», «yoʻlida» va «yoʻlidagi» mos kela boshlaydi. Tasodifiy gaplarda oʻsish ayniqsa katta, chunki ularda umumiy soʻzlar boshqa shakllarda turibdi.

## 6. Kalit soʻzlar: soʻzlarda TextRank

Xuddi shu algoritm soʻzlar uchun ham ishlaydi. Tugunlar — soʻz oʻzaklari, qirra toʻxtash soʻzlari olib tashlangandan keyin yonma-yon turgan (`window` oynasida) soʻzlarni bogʻlaydi. PageRank «qoʻshnilari» koʻp soʻzlarni tanlaydi. Asl TextRank da grafga faqat otlar va sifatlar olinadi; oʻzbek tili uchun tayyor POS-teggerlar kam, shuning uchun bu yerda toʻxtash soʻzlari bilan cheklanamiz.

In [ ]:
def keywords(sents, window=2, top=10):
    G = nx.Graph()
    surface = {}
    for s in sents:
        words = [w for w in tokens(s) if w not in STOP and len(w) > 2]
        stems = [stem(w) for w in words]
        for w, st in zip(words, stems):
            surface.setdefault(st, Counter())[w] += 1
        for i in range(len(stems)):                       # window dan yaqin turgan soʻzlarni bogʻlaymiz
            for j in range(i + 1, min(i + window, len(stems))):
                if stems[i] != stems[j]:
                    G.add_edge(stems[i], stems[j], weight=G.get_edge_data(stems[i], stems[j], {"weight": 0})["weight"] + 1)
    pr = nx.pagerank(G, weight="weight")
    best = sorted(pr, key=pr.get, reverse=True)[:top]
    return [(surface[st].most_common(1)[0][0], pr[st]) for st in best], G

kw, KG = keywords(sents, window=2)
print(f"soʻzlar grafi: {KG.number_of_nodes()} tugun, {KG.number_of_edges()} qirra")
for w, s in kw:
    print(f"   {w:<16} {s:.3f}")

## 7. Ekstraktiv va abstraktiv referat

**Ekstraktiv** usul gaplarni butunlay koʻchiradi, **abstraktiv** usul esa yangi matnni oʻz soʻzlari bilan yozadi: faktlarni birlashtiradi, qisqartiradi, qayta ifodalaydi. Bizning namuna abstraktiv. Uning qaysi soʻzlari asl matnda uchrashini (koʻchirilgan), qaysilari yangi ekanini (kvadrat qavsda — «yaratilgan») va referat bigrammlarining qancha qismi manbada yoʻqligini koʻramiz.

In [ ]:
source_stems = {stem(w) for w in tokens(text)}
abs_words = re.findall(r"\S+", reference)

copied = 0
line = []
for w in abs_words:
    toks = tokens(w)
    if not toks:                                   # tire va sonlarni hisoblamaymiz
        line.append(w)
        continue
    if all(stem(t) in source_stems for t in toks):
        line.append(w)
        copied += 1
    else:
        line.append("[" + w + "]")                 # soʻz manbada yoʻq: «yaratilgan»
print(" ".join(line))
total = sum(1 for w in abs_words if tokens(w))
print(f"\nmanbadan koʻchirilgan soʻzlar: {copied}/{total} = {copied / total:.0%}")

def bigrams(t):
    # har bir gap ichidagi qoʻshni soʻzlar juftlari
    return {b for s in split_sentences(t) for b in zip(tokens(s), tokens(s)[1:])}

src_bi = bigrams(text)
for name, cand in [("ekstraktiv", " ".join(summary)), ("abstraktiv", reference)]:
    b = bigrams(cand)
    print(f"yangi bigrammlar ({name}): {len(b - src_bi)}/{len(b)} = {len(b - src_bi) / len(b):.0%}")

Neyron abstraktiv modellar koʻpincha **pointer-generator** (See va boshq., 2017) sxemasida quriladi: har qadamda model soʻzni lugʻatdan yaratish yoki manbadan koʻchirishni hal qiladi. Yakuniy taqsimot ikkalasini aralashtiradi: $P(w) = p_{gen}\, P_{vocab}(w) + (1 - p_{gen}) \sum_{i: x_i = w} a_i$, bu yerda $a_i$ — manbaning $i$-soʻziga diqqat (attention). Quyidagi sonlar tasvir uchun oʻylab topilgan: model «Amir Temur Samarqandni oʻz davlatining ...» dan keyingi soʻzni tanlamoqda. Generatorning oʻzi «markaziga» ni afzal koʻrardi, lekin manbadan koʻchirish ustun keladi. `p_gen` ni oʻzgartirib koʻring (masalan, 0.9): qaysi qiymatda generatorning «oʻz» soʻzi yutadi?

In [ ]:
# pointer-generator: «lugʻatdan yaratish» va «manbadan koʻchirish» ni aralashtiramiz
p_gen = 0.3
p_vocab = {"markaziga": 0.40, "poytaxtiga": 0.25, "shahriga": 0.20, "qalbiga": 0.15}
p_attn = {"poytaxtiga": 0.75, "davlatining": 0.15, "Samarqandni": 0.10}
words = sorted(set(p_vocab) | set(p_attn))
P = {w: p_gen * p_vocab.get(w, 0) + (1 - p_gen) * p_attn.get(w, 0) for w in words}
for w in sorted(P, key=P.get, reverse=True):
    print(f"   {w:<14} {p_gen:.1f}·{p_vocab.get(w, 0):.2f} + {1 - p_gen:.1f}·{p_attn.get(w, 0):.2f} = {P[w]:.3f}")

### Haqiqiy modelda

`csebuetnlp/mT5_multilingual_XLSum` — XL-Sum yangiliklar korpusida (40 dan ortiq tildagi BBC maqolalari, ular orasida oʻzbek va rus tillari bor) qoʻshimcha oʻqitilgan mT5-base modeli. **Diqqat: model fayli ≈2.3 GB**, yuklash bir necha daqiqa davom etadi; Colab xotirasi yetadi. Model mualliflarining jadvalida oʻzbek tili uchun ROUGE-1 ≈ 16.8, rus tili uchun ≈ 32.2: XL-Sum da oʻzbekcha maqolalar taxminan 4,9 ming, ruscha esa 52,7 ming, shuning uchun oʻzbek tilida sifat pastroq. Model qaysi yozuvda (lotin yoki kirill) javob berishiga ham eʼtibor bering. Model yangilikning bir gapli «lid»ini yozishga oʻrgatilgan, shuning uchun referat qisqa boʻladi. Undagi har bir faktni asl matn bilan tekshiring: abstraktiv modellar baʼzan «gallyutsinatsiya» qiladi, yaʼni manbada yoʻq narsani yozadi. Kataklar faqat Colab da ishlaydi.

In [ ]:
!pip install -q sentencepiece

# haqiqiy model bilan abstraktiv referat (≈2.3 GB); generatsiya parametrlari model kartochkasidagidek
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    name = "csebuetnlp/mT5_multilingual_XLSum"
    tok = AutoTokenizer.from_pretrained(name)
    mt5 = AutoModelForSeq2SeqLM.from_pretrained(name)

    article = re.sub(r"\s+", " ", text.strip())
    inputs = tok([article], return_tensors="pt", truncation=True, max_length=512)
    output_ids = mt5.generate(**inputs, max_length=84, no_repeat_ngram_size=2, num_beams=4)[0]
    generated = tok.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    print("mT5:", generated)
    print("ROUGE-1 (P, R, F1) namunaga nisbatan:", tuple(round(x, 2) for x in rouge1(generated, reference, tokens)))
except Exception as e:
    print("Modelni yuklab boʻlmadi (internet, transformers va sentencepiece paketlari kerak):", e)

## 8. Topshiriqlar

1. **Qisqartmalar.** `split_sentences` ni shunday yaxshilangki, u «y.» qisqartmasi va ism-sharifning bosh harflaridan (nuqtali bitta bosh harf) keyin matnni boʻlmasin. 1-boʻlimdagi satrda tekshiring: 2 ta gap chiqishi kerak.
2. **Qirralar chegarasi.** Grafdan vazni `t` chegarasidan kichik qirralarni olib tashlang (`W * (W >= t)`) va PageRank ni `t = 0, 0.25, 0.3, 0.45` uchun qayta hisoblang. Eng yaxshi uchta gap qanday oʻzgaradi? Qirralari qolmagan tugunlar bilan nima boʻladi?
3. **d koeffitsienti.** PageRank ni `d = 0.5, 0.85, 0.99` uchun hisoblang. Yaqinlashish uchun nechta iteratsiya kerak? Baholar tarqoqligi (max − min) qanday oʻzgaradi?
4. **Siqish.** TextRank va Lead-k uchun ROUGE-1 $F_1$ (oʻzaklar boʻyicha) ning `ratio` ga (0.1 dan 0.6 gacha) bogʻliqlik grafigini chizing. Qaysi siqishda TextRank eng koʻp yutadi?
5. **Oʻz matningiz.** Boshqa mavzuda (masalan, Alisher Navoiy yoki oʻz shahringiz haqida) 10–15 gapli matn yozing va u uchun 3 gapli referat va 8 ta kalit soʻz oling. Maslahat: «Files» paneli orqali `my_text.txt` faylini yuklab, uni `open("my_text.txt", encoding="utf-8").read()` bilan oʻqishingiz mumkin.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing